# 05 — Iterative Refinement Demo

Demonstrates `IterativeRefinementOptimizer`: the LLM critiques its own prompt
each round and the best-matching static mutation is applied.  
Uses a deterministic mock LLM so no API key is needed to run this notebook.

In [ ]:
import sys
sys.path.insert(0, '../../')

from src.prompts.template import PromptTemplate
from src.search.iterative_refinement import IterativeRefinementOptimizer, _CRITIQUE_SAMPLE_QUESTION
from src.llm.base import BaseLLMClient
from src.utils.config_loader import load_search_config
from src.utils.visualization import plot_search_progress, plot_mutation_frequency

## Load Config

In [ ]:
cfg = load_search_config('iterative_refinement')
print('Config:', cfg)

## Mock LLM

- **Critique prompts** (contain the sample question): returns `"add_cot"` so the
  optimizer always picks the add_cot mutation.
- **Evaluation prompts**: returns `"YES"` only after add_cot is applied
  (detected by the `"step-by-step"` CoT trigger in the rendered prompt).

In [ ]:
class RefinementDemoLLM(BaseLLMClient):
    def __init__(self):
        self.call_count = 0

    def generate(self, prompt, temperature=0.0, max_tokens=500):
        self.call_count += 1
        if _CRITIQUE_SAMPLE_QUESTION in prompt:
            return 'add_cot — the prompt needs step-by-step reasoning to improve accuracy'
        return 'YES' if 'step-by-step' in prompt else 'NO'

    def get_usage_stats(self):
        return {'calls': self.call_count}

llm = RefinementDemoLLM()
print('Mock LLM ready')

## Validation Set

In [ ]:
VALIDATION_SET = [
    {'question': 'Can a student with MATH 171 and CPTS 121 take CPTS 223?', 'answer': 'YES'},
    {'question': 'Does completing CPTS 121 satisfy the prerequisite for CPTS 223?', 'answer': 'YES'},
    {'question': 'Is MATH 171 a prerequisite for MATH 172?', 'answer': 'YES'},
    {'question': 'Can a student enroll in CPTS 360 after CPTS 223 and MATH 216?', 'answer': 'YES'},
    {'question': 'Does a CS major need MATH 315 for graduation?', 'answer': 'YES'},
]
print(f'{len(VALIDATION_SET)} validation questions')

## Run Iterative Refinement

In [ ]:
initial_prompt = PromptTemplate(
    task_description='Answer the following WSU course advising question. Reply YES or NO.'
)

optimizer = IterativeRefinementOptimizer(
    llm_client=llm,
    max_rounds=cfg['max_rounds'],
    patience=cfg['patience'],
    accuracy_threshold=cfg['accuracy_threshold'],
)

best_prompt = optimizer.refine(initial_prompt, VALIDATION_SET)

print(f'Refinement complete — {len(optimizer.history)} round(s), {llm.call_count} API calls')

## Refinement Trace

In [ ]:
print(f'{"Round":>5}  {"Accuracy":>9}  {"Mutation Applied":>22}  Critique (truncated)')
print('-' * 90)
for entry in optimizer.history:
    critique_preview = (entry['critique'][:50] + '...') if len(entry['critique']) > 50 else entry['critique'] or '—'
    print(f"{entry['round']:>5}  {entry['accuracy']:>9.2%}  {entry['mutation_applied']:>22}  {critique_preview}")

## Convergence Curve

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_search_progress([optimizer.history], ['Iterative Refinement'], ax=axes[0])
plot_mutation_frequency(optimizer.history, ax=axes[1])
plt.tight_layout()
plt.show()

## Initial vs Final Prompt

In [ ]:
sample_q = VALIDATION_SET[0]['question']

print('=== Initial Prompt ===')
print(initial_prompt.render(sample_q))
print(f'\nMutation path: {initial_prompt.mutation_path()}')
print()
print('=== Final Refined Prompt ===')
print(best_prompt.render(sample_q))
print(f'\nMutation path: {best_prompt.mutation_path()}')

## Running with the Real Claude API

Uncomment below to run actual LLM critique rounds. The critique calls use
`critique_max_tokens=200` so they are cheap relative to full generation.

In [ ]:
# from dotenv import load_dotenv
# load_dotenv('../../.env')
#
# from src.llm.claude_client import ClaudeClient
# from src.llm.cache import ResponseCache
#
# real_llm = ClaudeClient(cache=ResponseCache())
# real_optimizer = IterativeRefinementOptimizer(
#     llm_client=real_llm,
#     max_rounds=cfg['max_rounds'],
#     patience=cfg['patience'],
#     accuracy_threshold=cfg['accuracy_threshold'],
# )
# real_best = real_optimizer.refine(initial_prompt, VALIDATION_SET)
# print('Best path:', real_best.mutation_path())